In [121]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [122]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-05-16,99.522102,99.885457,98.348183,98.441357
1,2016-05-17,98.273674,99.727093,98.012797,99.438275
2,2016-05-18,98.627708,99.158766,97.863731,98.050064
3,2016-05-19,98.115295,98.487969,97.397904,98.245730
4,2016-05-20,99.196030,99.596653,98.450682,98.506588
...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022


In [123]:
df['EMA10'] = df.Close.ewm(span=10).mean()
df['EMA30'] = df.Close.ewm(span=30).mean()

def RSI(data, n):
    pc = df['Close'].diff()
    up = pc.clip(lower=0)
    dn = pc.clip(upper=0).abs()
    avg_up = up.ewm(alpha=1/n, adjust=False).mean()
    avg_dn = dn.ewm(alpha=1/n, adjust=False).mean()
    RS = avg_up / avg_dn
    RSI = 100 - (100 / (1 + RS))
    return RSI

df['RSI8'] = RSI(df, 8)
df.dropna(inplace=True)
df
    


Price,Date,Close,High,Low,Open,EMA10,EMA30,RSI8
1,2016-05-17,98.273674,99.727093,98.012797,99.438275,98.835467,98.877081,0.000000
2,2016-05-18,98.627708,99.158766,97.863731,98.050064,98.751949,98.788356,3.893469
3,2016-05-19,98.115295,98.487969,97.397904,98.245730,98.542200,98.602903,3.657891
4,2016-05-20,99.196030,99.596653,98.450682,98.506588,98.729897,98.737854,15.920386
5,2016-05-23,99.065613,99.661888,98.972446,99.307845,98.817094,98.801975,15.645745
...,...,...,...,...,...,...,...,...
2509,2026-05-08,711.229980,711.229980,699.500000,699.919983,681.152151,649.476255,89.194065
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,686.995392,653.593270,89.609585
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,690.676228,657.054348,79.366171
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,695.046009,660.774069,82.232218


In [124]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.EMA10.iloc[i] > data.EMA30.iloc[i]) and (data.RSI8.iloc[i-1] < 50) and (data.RSI8.iloc[i] > 50):
            signal[i] = 1
        elif (data.EMA10.iloc[i] < data.EMA30.iloc[i]) and (data.RSI8.iloc[i-1] > 50) and (data.RSI8.iloc[i] < 50):
            signal[i] = 2
        elif (data.RSI8.iloc[i-1] < 70) and (data.RSI8.iloc[i] > 70):
            signal[i] = 3
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)

In [125]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def more_long_entries(x):
    offset = 0.002
    if x['signal']==3:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['more_long_entries'] = df.apply(lambda x: more_long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    2237
3     136
1     101
2      39
Name: count, dtype: int64


(2513, 12)

In [126]:
df.set_index('Date', inplace=True)

In [127]:
bar = 2170
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    row_heights=[0.68,0.32], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['more_long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="blue"),
                name="More Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.EMA10, 
                         opacity=0.8, 
                         line=dict(color='white', width=1),
                        name='EMA10'))

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.EMA30, 
                         opacity=0.8, 
                         line=dict(color='gold', width=1),
                        name='EMA30'))


fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.RSI8, 
                         line=dict(color='red', width=2),
                         name='RSI8'),
                         row=2, col=1)

fig.add_hline(y=70, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=30, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.update_layout(autosize=False, width=1100, height=700, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()








In [131]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, sl=0.98*price, tp=1.06*price)

        elif self.signal==3: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.015*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-05-17 00:00:00
End                       2026-05-14 00:00:00
Duration                   3649 days 00:00:00
Exposure Time [%]                    49.18424
Equity Final [$]                1312339.63535
Equity Peak [$]                  1574834.9389
Commissions [$]                  120230.27075
Return [%]                         1212.33964
Buy & Hold Return [%]               633.53329
Return (Ann.) [%]                    29.45417
Volatility (Ann.) [%]                30.20689
CAGR [%]                             19.45719
Sharpe Ratio                          0.97508
Sortino Ratio                         1.87048
Calmar Ratio                            0.756
Alpha [%]                           884.68727
Beta                                  0.51718
Max. Drawdown [%]                   -38.96045
Avg. Drawdown [%]                    -3.65738
Max. Drawdown Duration      715 days 00:00:00
Avg. Drawdown Duration       25 days 00:00:00
# Trades                          

In [129]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [130]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='Equity',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()